In [ ]:
import tensorflow as tf
import numpy as np
import pandas as pd
from tensorflow.keras import layers, models, losses 
from tensorflow.keras.datasets import mnist 
from tensorflow.keras.metrics import Precision, Recall
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
from scipy.ndimage import rotate

In [ ]:
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.mnist.load_data()
# imgs
print(x_train.shape == (60000, 28, 28))
print(x_test.shape == (10000, 28, 28))
# labels 
print(y_train.shape == (60000,))
print(y_test.shape == (10000,))

In [ ]:
angles = [i for i in range(-20, 21, 5)]
NOISE_STD = 20.0
new_x = []
new_y = []

for img, label in zip(x_train, y_train):
    # without augmentation
    new_x.append(img)
    new_y.append(label)

    for angle in angles:
        if angle == 0:
            noise = np.random.normal(loc=0.0, scale=NOISE_STD, size=img.shape)
            img_noised = img + noise
            img_noised = np.clip(img_noised, 0, 255).astype(np.uint8)
            new_x.append(img_noised)
            new_y.append(label)
            continue

        img_rot = rotate(img, angle, axes=(0,1), reshape=False, mode='constant', cval=0.0)
        new_x.append(img_rot)
        new_y.append(label)
        noise = np.random.normal(loc=0.0, scale=NOISE_STD, size=img_rot.shape)
        img_noised = img_rot + noise
        img_noised = np.clip(img_noised, 0, 255).astype(np.uint8)

        new_x.append(img_noised)
        new_y.append(label)

x_train_augmented = np.array(new_x)
y_train_augmented = np.array(new_y)


In [ ]:
model = models.Sequential([
    layers.Input((28,28,1)),
    layers.Rescaling(1./255),
    layers.Conv2D(32, 3, activation='relu'),
    layers.MaxPooling2D(pool_size=(2)),
    layers.Conv2D(64, 3, activation='relu'),
    layers.MaxPooling2D(pool_size=(2)),
    layers.Flatten(),
    layers.Dense(64, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(10, activation='softmax')
])

In [ ]:
model.compile(optimizer='adam', loss=losses.SparseCategoricalCrossentropy(), metrics=['accuracy'],)
model.fit(x=x_train_augmented, y=y_train_augmented, batch_size=64, epochs=1, validation_data=(x_test, y_test))

In [ ]:
model_loss, model_accuracy = model.evaluate(x_test, y_test)

In [ ]:
print(f"Loss: {model_loss}")
print(f"Accuracy: {model_accuracy*100:.2f}")

In [ ]:
predictions = model.predict(x_test)
decisions = np.argmax(predictions, axis=1)

In [ ]:
print(classification_report(y_test, decisions))

In [ ]:
cm = confusion_matrix(y_test, decisions)
df_cm = pd.DataFrame(cm)
df_cm.index.name = "Real"
df_cm.columns.name = "Predicted"
df_cm

In [ ]:
model.save('../models/digit-recognizer-with-augmentation.keras')